PART 1: 

In [2]:
import pandas as pd
import numpy as np 
df = pd.read_csv("../data/student_performance.csv")

We are using the student_performance.csv . The response variable is final_score. The predictor variables are study_hours_per_week , attendance_pct , and prior_score. Our goal is to fit and diagnose a linear model using matrix compuation.

In [3]:
#DATA INSPECTION
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df["id"].is_unique)



   id  study_hours_per_week  attendance_pct  prior_score  final_score
0   1                  5.00           70.44        69.31        84.63
1   2                  7.12           74.32        79.58       102.25
2   3                  4.79           77.05        95.89        94.66
3   4                  6.15           81.81        82.36        92.11
4   5                  8.35           69.14        64.04        85.30
(120, 5)
id                        int64
study_hours_per_week    float64
attendance_pct          float64
prior_score             float64
final_score             float64
dtype: object
id                      0
study_hours_per_week    0
attendance_pct          0
prior_score             0
final_score             0
dtype: int64
True


The dataset contains 120 observations and 5 variables. There are no missing values in any of the columns. Amzazing! Additionally, the ID validation verifies that all student IDs are unique This data looks clean and is ready for matrix operations and further analytics without extra processing.

PART 2:

In [4]:
#We are hoping to build out the pieces we will need to build our linear model.
pred = df[["study_hours_per_week", "attendance_pct", "prior_score"]].to_numpy()
y = df["final_score"].to_numpy()

means = pred.mean(axis=0)
sds = pred.std(axis=0)
Z = (pred - means) / sds

#Assuming my standardization worked out, we should see N ~ (0 , 1) or mean of 0 and standard deviation of 1.
print("Z means:", Z.mean(axis=0))
print("Z Standard Deviation", Z.std(axis=0))

X = np.column_stack([np.ones(len(Z)), Z])


print("Shape of Predictors:", pred.shape)
print("Shape of Z:", Z.shape)
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Matrix Rank for X:", np.linalg.matrix_rank(X))
print("Number of predictors excluding intercept", pred.shape[1])


Z means: [ 3.70074342e-17 -7.47550170e-16  1.50250183e-15]
Z Standard Deviation [1. 1. 1.]
Shape of Predictors: (120, 3)
Shape of Z: (120, 3)
Shape of X: (120, 4)
Shape of y: (120,)
Matrix Rank for X: 4
Number of predictors excluding intercept 3


X and Y must have compatible shapes as they will be used to perform matrix computations, noteablly, beta(hat) =(XᵀX)⁻¹Xᵀy. These matrices must have compatible row dimensions. The intercept column of ones is included in X so that we can have a beta(naught) term and establish a baseline expected score. Standardizing the predictors allows us to compare the magnitude of the resulting coefficients on a uniform scale, and it improves the numerical stability of the matrix inversion. Broadcasting lets numpy implicitly expand 1D arrays of means and standard deviations to match the 2D predictor shape. This lets us complete element-wise operations, like standardization, for each row simultaneously without using a loop. Nice.


Part 3:

In [5]:
beta_hat, _, _, _ = np.linalg.lstsq(X, y, rcond=None)

print("Estimated coefficients:")
print(f"Intercept: {beta_hat[0]:.4f}")
print(f"Study Hours: {beta_hat[1]:.4f}")
print(f"Attendance: {beta_hat[2]:.4f}")
print(f"Prior Score: {beta_hat[3]:.4f}")

fitted = X @ beta_hat
residuals = y - fitted

print(f"Residual Mean: {residuals.mean()}") #We would hope to see residual mean at 0 or close to it. We don't want to see trends or patterns in residual means.

rmse = np.sqrt(np.mean(residuals **2))

print(f"RMSE: {rmse: .4f}")
print(f"Prior score has the largest coefficient in absolute value {beta_hat[3]: .4f}")

Estimated coefficients:
Intercept: 92.9125
Study Hours: 2.9075
Attendance: 0.9388
Prior Score: 3.2297
Residual Mean: -2.7948014273230606e-14
RMSE:  4.9777
Prior score has the largest coefficient in absolute value  3.2297


X @ beta_hat calculates the predicted values of our linear model. The residual mean should be near 0 since the fitted line passes exactly through the mean of the data when an intercept is present. This means deviations from the fitted line are balanced above and below the line, causing the sum and mean of residuals to equal 0. RMSE summarizes the standard deviation of the residuals, measuring the typical error between our model and the observed values. Finally, the difference between X * beta_hat and X @ beta_hat is that the first performs element-wise multiplication. The second computes formal matrix multiplication. 

Part 4:

In [6]:
#We are trying to see what is going on behind the curtain of lstsq().
Q, R = np.linalg.qr(X)
print("Shape of Q:", Q.shape)
print("Sahpe of R:", R.shape)
#Q is a orthogonal Matrix, R is an upper trianglar matirx
print("Q @ R, X:", np.allclose(Q @ R, X))
print("Q.T @ Q:", np.allclose(Q.T @ Q, np.eye(Q.shape[1])))

#SVD Decomposition (singlar value decomposition)
U, s, Vt = np.linalg.svd(X, full_matrices=False)
print("\nShape of U:", U.shape)
print("Shape of s:", s.shape)
print("Shape of Vt:", Vt.shape)
print("Singular Values (s)", s) #If we see s values close to 0, this may indicate redundant variables. You want higher values here for each of the s's. 
#Reconstruct X
X_reconstructed = U @ np.diag(s) @ Vt 
print("U @ diag(s) @ Vt:", np.allclose(X_reconstructed, X)) #reconstructed X verifies that not info was lost in our decomposition.

#Rank & Conditioning (Standardized)
print("\nStandardized:")
print("Rank of X:", np.linalg.matrix_rank(X))
print("Condition Number of X:", np.linalg.cond(X))
print("Raw:")
X_raw = np.column_stack([np.ones(len(pred)), pred])
print("Condition Number of X_raw:", np.linalg.cond(X_raw))




Shape of Q: (120, 4)
Sahpe of R: (4, 4)
Q @ R, X: True
Q.T @ Q: True

Shape of U: (120, 4)
Shape of s: (4,)
Shape of Vt: (4, 4)
Singular Values (s) [13.34687885 11.1939613  10.95445115  7.520376  ]
U @ diag(s) @ Vt: True

Standardized:
Rank of X: 4
Condition Number of X: 1.7747621723767468
Raw:
Condition Number of X_raw: 1484.16939639802


Q @ R ≈ X shows that the QR decomposition successfully reconstructs the original design matrix without losing information contained in the data. Q.T @ Q ≈ I verifies that Q is an orthogonal matrix, where its columns are perpendicular unit vectors. The singular values in the SVD tell us the magnitude of the data's spread. A mixture of large s values mixed with smaller, near-zero s values would likely indicate collinearity. A large condition number suggests the matrix is very sensitive to numerical errors. The condition number dropped from roughly 1484.17 to 1.77 after standardization. Measurement scale strongly affects the raw condition number because mixing large units and small units can skew the singular values, creating numerical instability. Since our standardized condition number is so small, the standardized design matrix appears to show no evidence of hard redundancy with the predictors. This is pretty crazy!

Part 5:

(1-2) Clean execution is extremely important. It allows for independent users to run the Python code from start to finish without having to manually intervene and add more code to allow this to run. Reusable code should never rely on a hidden notebook state. This would include things like variables held in memory from out-of-order cell execution. This could allow for inaccurate information to be given that an independent user might not be able to detect as easily as the author of the environment.

(3-5) Utilization of the np.linalg.lstsq() is far more preferable to computing (XᵀX)⁻¹Xᵀy and completing this computation essentially follows the same mathematical procedure that one would use to solve this out by hand. Your computer will work through one-by-one or each matrix value in order to complete the action. Luckily for us, the lstsq() function goes around the inversion process by using orthogonal decompositions to solve out a system with much higher numerical stability. Checks like np.allclose() are preferable to exact equality for QR and SVD reconstructions due to the exact equality failing due to extremely small rounding errors. This allows us to evaluate equivalence within a certain tolerance essentially since our computer shouldn't have to store almost infinite decimal places that could be associated with certain values. Condition-number comparison taught me how amazing the mathematics behind stats is. Standardization is an amazing tool. We could see that the raw condition vs the standardized condition yielded a much higher, more variable number of 1484. However, this is due to the difference in measurement units among the predictors. After standardization, we can now compare these on the same level of magnitude. This allows for a much more stable 1.77 standardized condition number.

(6) There are a few reproducibility factors from this homework, as well as prior homeworks, that I will carry forward to the final project. First, I will use relative paths. This is huge for allowing independent users to reproduce results from the code I have generated. I will work to isolate loaded data using a dedicated directory like data/raw or data/processed. I also find it vitally important to not include changes of variables or hidden notebook states as the code progresses forward. This may allow for the code to work but produce incorrect numbers for an independent user.

